# 0. 导入必要的库

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
import torchvision
from torchvision.transforms import ToPILImage
from typing import List, Tuple, Optional, Dict, Any

# Import utility functions
import sys
sys.path.append('.')
from my_utils import (
    CIFAR_10_MEAN,
    CIFAR_10_STD,
    CIFAR_10_CLASS,
    get_device,
    get_cifar10_data_augmentation,
    create_train_val_split,
    train_model,
    predict as predict_util,
    plot_loss_curves,
    count_parameters,
    create_learning_rate_scheduler
)

show = ToPILImage()

# 1. 数据准备

In [ ]:
# Set image normalization transforms and download the dataset
os.makedirs('./dataset', exist_ok=True)

# Use utility functions to get data augmentation transforms
transform_train, transform_test = get_cifar10_data_augmentation()

batch_size = 64  # Set batch size to 64 for RTX 4090

# Load full training set
full_trainset = torchvision.datasets.CIFAR10(
    root='./dataset', 
    train=True,
    download=True, 
    transform=transform_train
)

# Split training set into training and validation sets (90% train, 10% validation)
trainset, valset = create_train_val_split(
    dataset=full_trainset, 
    val_ratio=0.1, 
    seed=51
)

# Test set
testset = torchvision.datasets.CIFAR10(
    root='./dataset', 
    train=False,
    download=True, 
    transform=transform_test
)

# Create data loaders
trainloader = torch.utils.data.DataLoader(
    dataset=trainset, 
    batch_size=batch_size,
    shuffle=True, 
    num_workers=2
)

valloader = torch.utils.data.DataLoader(
    dataset=valset, 
    batch_size=batch_size,
    shuffle=False, 
    num_workers=2
)

testloader = torch.utils.data.DataLoader(
    dataset=testset, 
    batch_size=batch_size,
    shuffle=False, 
    num_workers=2
)

/root/Snape/Introduction-to-AI-Project-01/venv/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train set size: 45000, Validation set size: 5000
Training set size: 45000
Validation set size: 5000
Test set size: 10000


In [ ]:
# Data Exploration: Class Distribution and Dataset Statistics
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
FIGURE_DIR = Path('figures')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
save_path = FIGURE_DIR / 'data_exploration_class_distribution.png'

# Collect labels from the full training set
train_labels = [full_trainset[i][1] for i in range(len(full_trainset))]
class_counts = [train_labels.count(i) for i in range(10)]

print("=" * 60)
print("CIFAR-10 Dataset Statistics")
print("=" * 60)
print(f"Full training set:  {len(full_trainset):>5} images")
print(f"  - Training split: {len(trainset):>5} images")
print(f"  - Validation split:{len(valset):>5} images")
print(f"Test set:           {len(testset):>5} images")
print(f"Image dimensions:   {full_trainset[0][0].shape}")
print(f"Number of CIFAR_10_CLASS:  {len(CIFAR_10_CLASS)}")
print()
print("Class Distribution:")
print("-" * 40)
for i, (cls, count) in enumerate(zip(CIFAR_10_CLASS, class_counts)):
    print(f"  {i}: {cls:<8s}  {count:>5} images ({100 * count / len(full_trainset):.1f}%)")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
colors = plt.cm.tab10(np.arange(10))

ax1.bar(CIFAR_10_CLASS, class_counts, color=colors, edgecolor='white', linewidth=1.5)
ax1.set_title('Class Distribution', fontsize=13, fontweight='bold')
ax1.set_xlabel('Class')
ax1.set_ylabel('Number of Samples')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

ax2.pie(class_counts, labels=CIFAR_10_CLASS, autopct='%1.1f%%', colors=colors,
        startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1})
ax2.set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Data Exploration: Sample Grid from Each Class
save_path = FIGURE_DIR / 'data_exploration_sample_grid.png'
samples_per_class = 5

# Load raw training set (without augmentation for clean display)
raw_trainset = torchvision.datasets.CIFAR10(
    root='./dataset', 
    train=True, 
    download=False,
    transform=torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(CIFAR_10_MEAN, CIFAR_10_STD)
    ])
)

# Collect images per class
class_samples = {i: [] for i in range(10)}
for img, label in raw_trainset:
    if len(class_samples[label]) < samples_per_class:
        class_samples[label].append(img)
    if all(len(v) >= samples_per_class for v in class_samples.values()):
        break

# Plot grid: 10 rows (CIFAR_10_CLASS) x 5 columns (samples)
fig, axes = plt.subplots(10, samples_per_class, figsize=(samples_per_class * 2, 20))
fig.suptitle('CIFAR-10: Sample Images from Each Class', fontsize=16, fontweight='bold', y=1)

mean_t = torch.tensor(CIFAR_10_MEAN).view(3, 1, 1)
std_t = torch.tensor(CIFAR_10_STD).view(3, 1, 1)

for i in range(10):
    for j in range(samples_per_class):
        img = class_samples[i][j]
        img_display = img * std_t + mean_t  # Denormalize
        axes[i, j].imshow(img_display.permute(1, 2, 0).clamp(0, 1).numpy())
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_ylabel(CIFAR_10_CLASS[i], fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Data Exploration: Per-Channel Pixel Statistics and Histograms
import torchvision
import numpy as np
import matplotlib.pyplot as plt

# Load raw training set (ToTensor only, no normalization)
raw_trainset_stats = torchvision.datasets.CIFAR10(
    root='./dataset', train=True, download=False,
    transform=torchvision.transforms.ToTensor()
)

# Stack images (use first 10000 for speed, statistics are representative)
all_pixels = np.stack([np.array(raw_trainset_stats[i][0]) for i in range(10000)])

print("=" * 52)
print("Per-Channel Pixel Statistics (raw [0, 1] range)")
print("=" * 52)
channels = ['Red', 'Green', 'Blue']
for i, ch_name in enumerate(channels):
    ch = all_pixels[:, i, :, :]
    print(f"{ch_name:>6} | Mean: {ch.mean():.4f} | Std: {ch.std():.4f} | "
          f"Min: {ch.min():.4f} | Max: {ch.max():.4f}")

print()
print(f"Overall   | Mean: {all_pixels.mean():.4f} | Std: {all_pixels.std():.4f}")

# Histograms
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors_hist = ['#e74c3c', '#2ecc71', '#3498db']

for i, (ch_name, color) in enumerate(zip(channels, colors_hist)):
    ch_data = all_pixels[:, i, :, :].flatten()
    axes[i].hist(ch_data, bins=60, color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
    axes[i].axvline(ch_data.mean(), color='black', linestyle='--', linewidth=1.5,
                    label=f'Mean={ch_data.mean():.3f}')
    axes[i].set_title(f'{ch_name} Channel', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Pixel Value')
    axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('CIFAR-10 Pixel Value Distribution (per Channel)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data_exploration_pixel_histograms.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Data Exploration: Data Augmentation Visualization
import numpy as np
import matplotlib.pyplot as plt

# Get a clean PIL image and its label
raw_pil_dataset = torchvision.datasets.CIFAR10(root='./dataset', train=True, download=False)
sample_img, sample_label = raw_pil_dataset[0]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

mean_t = torch.tensor(CIFAR_10_MEAN).view(3, 1, 1)
std_t = torch.tensor(CIFAR_10_STD).view(3, 1, 1)

# Original image (no augmentation)
axes[0].imshow(np.array(sample_img))
axes[0].set_title(f'Original ({CIFAR_10_CLASS[sample_label]})', fontsize=11, fontweight='bold')
axes[0].axis('off')

# 7 augmented versions using transform_train (defined in section 1, already uses CIFAR_10_MEAN/CIFAR_10_STD)
for i in range(1, 8):
    aug = transform_train(sample_img)
    aug_display = aug * std_t + mean_t  # Denormalize
    axes[i].imshow(aug_display.permute(1, 2, 0).clamp(0, 1).numpy())
    axes[i].set_title(f'Augmented {i}', fontsize=11, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Data Augmentation Examples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data_exploration_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Augmentation visualization saved. Class: {CIFAR_10_CLASS[sample_label]}")

# 2. 定义用于分类的网络结构

这一部分我们定义用于图像分类的网络结构，实现一个早期的卷积神经网络LeNet。它由两个卷积层和三个全连接层组成。pytorch为我们提供了方便的接口定义神经网络，但我们这里不着重介绍具体的语法，只观察数据是怎样在模型中“流动”的：
- 在`__init__`方法中，我们将上述的卷积层和全连接层初始化为`conv1、conv2`和`fc1、fc2、fc3`；
- 卷积层以`conv1`为例，它的初始化为`Conv2d(3, 6, 5)`，即：3输入通道（RGB图像的三个通道）、6输出通道、5*5大小的卷积核的卷积层。
- 全连接层以`fc1`为例，它的初始化为`Linear(16 * 5 * 5, 120)`，即：从400维映射到120维。
- `forward`方法用于规定数据在模型中的计算过程。输入的形状在传播过程中的变化参见`forward`中的注释。最终，我们得到了一个大小为`[batch size, 10]`的张量（矩阵）。

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    """LeNet-style CNN for CIFAR-10 image classification.
    
    Original architecture adapted for 32×32 RGB images.
    Consists of two convolutional layers and three fully connected layers.
    """
    
    def __init__(self) -> None:
        # Subclasses of nn.Module must call the parent class constructor
        super(Net, self).__init__()

        # Convolutional layer: 3 input channels (RGB), 6 output channels, 5x5 kernel
        self.conv1 = nn.Conv2d(3, 6, 5) 
        # Convolutional layer
        self.conv2 = nn.Conv2d(6, 16, 5) 
        # Fully connected layers: y = Wx + b
        self.fc1   = nn.Linear(16*5*5, 120) 
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the network.
        
        Args:
            x: Input tensor of shape (batch_size, 3, 32, 32).
            
        Returns:
            Output tensor of shape (batch_size, 10).
        """
        # Convolution -> ReLU -> MaxPool (ReLU does not change shape)
        # [batch_size, 3, 32, 32] -- conv1 --> [batch_size, 6, 28, 28] -- maxpool --> [batch_size, 6, 14, 14]
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        # [batch_size, 6, 14, 14] -- conv2 --> [batch_size, 16, 10, 10] -- maxpool --> [batch_size, 16, 5, 5]
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        # Flatten the 16 * 5 * 5 feature map into [batch_size, 16 * 5 * 5] for the fully connected layers
        x = x.view(x.size()[0], -1) 
        # [batch_size, 16 * 5 * 5] -- fc1 --> [batch_size, 120]
        x = F.relu(self.fc1(x))
        # [batch_size, 120] -- fc2 --> [batch_size, 84]
        x = F.relu(self.fc2(x))
        # [batch_size, 84] -- fc3 --> [batch_size, 10]
        x = self.fc3(x)        
        return x

net = Net()
print(net)

# 3. 模型训练与测试过程
准备好数据、定义好模型后，我们开始训练过程。为了把一个随机初始化的模型优化成一个“好”的模型，我们还需要定义：
- 损失函数$\mathcal{L}$：损失函数以一般同时以模型的预测$\hat{y}$和真实的标签$y$为输入，输出一个标量。这个标量越小，说明模型在数据上拟合得越好。我们的目的就是要最小化这个损失函数$\mathcal{L}(\hat{y},y).$分类问题常使用交叉熵函数作为损失函数。
- 优化方法：为了最小化损失函数，我们就要使用数学的优化方法找到一组最优的参数（这里的参数即神经网络中卷积层、全连接层等的参数，而非batch size等超参数）。深度学习中一般使用迭代的方式求解，常用的方法有SGD（随机梯度下降）、Adam等。
pytorch库内置了各种优化器，我们无需手动实现梯度下降过程。

In [ ]:
from torch import optim

# Loss function: supports label smoothing
label_smoothing = 0.0  # Default: no label smoothing; will be adjusted in Task3
criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

# Optimizer: SGD with momentum (baseline)
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9, weight_decay=0.0)  # No L2 regularization

# Number of training epochs
num_epochs = 20  # Increased from 5 to 20, with early stopping

print(f"Loss function: CrossEntropyLoss (label_smoothing={label_smoothing})")
print(f"Optimizer: SGD (lr=0.001, momentum=0.9, weight_decay=0.0)")
print(f"Epochs: {num_epochs} (with early stopping)")

下面我们定义用于训练过程的代码。最外层循环控制在整个数据集上训练的次数（即epoch）；内层循环按照以下流程进行：
1. 取出数据（一次取出一个batch）；
2. 将数据送入网络，计算损失函数；
3. 使用损失函数计算梯度，进行反向传播更新参数。

In [ ]:
def train(
    trainloader: torch.utils.data.DataLoader,
    net: nn.Module,
    num_epochs: int,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    save_path: str,
    valloader: Optional[torch.utils.data.DataLoader] = None,
    early_stopping_patience: Optional[int] = None,
    scheduler_config: Optional[Dict[str, Any]] = None
) -> Tuple[List[float], Optional[List[float]]]:
    """Wrapper around my_utils.train_model for training a neural network.
    
    Args:
        trainloader: DataLoader for training data.
        net: Neural network model.
        num_epochs: Number of training epochs.
        criterion: Loss function.
        optimizer: Optimizer.
        save_path: Directory to save model checkpoints.
        valloader: DataLoader for validation data (optional).
        early_stopping_patience: Patience for early stopping (optional).
        scheduler_config: Learning rate scheduler config from
            create_learning_rate_scheduler (optional).
    
    Returns:
        train_losses: List of average training loss per epoch.
        val_accuracies: List of validation accuracy per epoch (if valloader provided).
    """
    print("Note: Using my_utils.train_model for training")
    return train_model(
        model=net,
        train_loader=trainloader,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=num_epochs,
        save_path=save_path,
        val_loader=valloader,
        early_stopping_patience=early_stopping_patience,
        gradient_clip=1.0,
        print_every=1000,
        scheduler_config=scheduler_config
    )

In [ ]:
# Set device (GPU if available)
device = get_device()
print(f"Using device: {device}")

# Move model to device
net = Net()
net = net.to(device)

# Create learning rate scheduler config (linear warm-up + cosine annealing with restarts)
scheduler_config = create_learning_rate_scheduler(
    optimizer, scheduler_type='cosine',
    total_epochs=num_epochs, initial_lr=0.001
)

# Train the network
save_path = 'checkpoints/lenet_baseline'
train_losses, val_accuracies = train(
    trainloader, net, num_epochs, criterion, optimizer, save_path,
    valloader=valloader, early_stopping_patience=10,
    scheduler_config=scheduler_config
)

# Plot loss curves using utility function
plot_loss_curves(train_losses, val_accuracies, 'loss_curve.png')

print(f"Final training loss: {train_losses[-1]:.4f}")
if val_accuracies:
    print(f"Final validation accuracy: {val_accuracies[-1]:.2f}%")

训练过程结束后，我们得到了一个在训练集上拟合较好的模型。下面我们要测试它在测试集上表现如何。预测的代码与训练中的正向传播类似，但是不需要计算损失函数（损失函数在实验中仅用于更新参数，预测时参数固定，也就不需要它了）。

预测的流程如下：
1. 取出数据；
2. 正向传播，得到模型的输出结果；
3. 从输出结果中得到模型预测；
4. 和真实标签进行比对，计算性能指标。

注意：模型的输出结果在第2部分中已经说明，为一个`[batch size, 10]`大小的张量（矩阵），每一行是一条数据属于10个类别的概率的相对大小（这一输出也被称为`logits`）。为了得到模型的预测，我们需要对这一输出在每行上取最大值，取得最大值的**位置**就是模型的预测。

In [ ]:
def predict(testloader: torch.utils.data.DataLoader, net: nn.Module) -> float:
    """Predict and calculate test accuracy (wrapper around my_utils.predict).
    
    Args:
        testloader: DataLoader for test data.
        net: Neural network model.
    
    Returns:
        accuracy: Test accuracy percentage (0-100).
    """
    print("Note: Using my_utils.predict for evaluation")
    return predict_util(net, testloader)

In [20]:
predict(testloader, net)

测试集中的准确率为: 59 %



# Task1：绘制损失函数曲线
损失函数能够量化模型在数据集上的拟合程度，帮助我们了解模型训练的进程。请在`3.模型训练与测试过程`中补充代码，记录训练过程中损失`loss`的变化，使用合适的Python数据类型将其保存，并使用`matplotlib`库将其可视化。可参照以下的代码进行绘图。你可以直接用损失函数可视化的代码覆盖下面的代码块。

In [ ]:
# Task1: Plot Training Loss Curve
# Using plot_loss_curves from my_utils

print("Task1: Training Loss Curve Visualization")
print("="*50)
print("The plot_loss_curves function has been imported from my_utils.")
print("Usage:")
print("  plot_loss_curves(train_losses, val_accuracies, 'task1_loss_curve.png')")
print("")
print("Function signature:")
print("  plot_loss_curves(train_losses, val_accuracies=None, save_path='loss_curve.png')")
print("")
print("Parameters:")
print("  - train_losses: List of training losses per epoch")
print("  - val_accuracies: List of validation accuracies per epoch (optional)")
print("  - save_path: Path to save the plot image")
print("")
print("plot_loss_curves has already been called in the training cell above.")

请在报告中附上训练过程中损失函数的变化。训练集上的损失越小，说明模型的效果就越好吗？

# Task2: 加入正则化

- $L_2$正则化：请查阅Pytorch[有关SGD优化器的文档](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html#sgd)或其它网络资料，修改`3. 模型训练与测试过程`中的代码，尝试为模型的损失函数加入一项$L_2$损失，并在报告中说明你所做的修改。
- Dropout正则化：请查阅Pytorch[有关Dropout层的文档](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html#dropout)或其它网络资料，修改`2. 定义用于分类的网络结构`中的代码，在**第一个线性层和第二个线性层之间**加入一个Dropout层，并在报告中说明你所做的修改。
- 在报告中简述两种正则化方法的基本原理。

In [ ]:
# Dropout_Net: LeNet with dropout layer added
class Dropout_Net(nn.Module):
    """LeNet variant with dropout regularization.
    
    Adds dropout layer between fc1 and fc2 to prevent overfitting.
    """
    
    def __init__(self, dropout_rate: float = 0.5) -> None:
        # Subclasses of nn.Module must call the parent class constructor
        super(Dropout_Net, self).__init__()

        # Convolutional layer: 3 input channels (RGB), 6 output channels, 5x5 kernel
        self.conv1 = nn.Conv2d(3, 6, 5) 
        # Convolutional layer
        self.conv2 = nn.Conv2d(6, 16, 5) 
        # Fully connected layers: y = Wx + b
        self.fc1   = nn.Linear(16*5*5, 120) 
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, 10)
        
        # Dropout layer
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass with dropout between fc1 and fc2.
        
        Args:
            x: Input tensor of shape (batch_size, 3, 32, 32).
            
        Returns:
            Output tensor of shape (batch_size, 10).
        """
        # Convolution -> ReLU -> MaxPool
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        x = F.max_pool2d(F.relu(self.conv2(x)), 2) 
        # Flatten; '-1' means automatically inferred size
        x = x.view(x.size()[0], -1) 
        x = F.relu(self.fc1(x))
        # Dropout between fc1 and fc2
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)        
        return x
    
dropout_net = Dropout_Net(dropout_rate=0.5)
print(dropout_net)

In [ ]:
# Task2: Train model with L2 regularization and Dropout

# Set device
device = get_device()

# Create Dropout_Net instance and move to device
dropout_net = Dropout_Net(dropout_rate=0.5)
dropout_net = dropout_net.to(device)

# Define optimizer with L2 regularization (via weight_decay)
# Using Adam optimizer; weight_decay implements L2 regularization
optimizer_dropout = torch.optim.Adam(dropout_net.parameters(), lr=0.001, weight_decay=0.001)

# Create learning rate scheduler (linear warm-up + cosine annealing with restarts)
scheduler_config_dropout = create_learning_rate_scheduler(
    optimizer_dropout, scheduler_type='cosine',
    total_epochs=num_epochs, initial_lr=0.001
)

# Train Dropout network
save_path_dropout = 'checkpoints/lenet_dropout'
print("Training Dropout_Net with L2 regularization...")
train_losses_dropout, val_accuracies_dropout = train(
    trainloader, dropout_net, num_epochs, criterion, optimizer_dropout, 
    save_path_dropout, valloader=valloader, early_stopping_patience=10,
    scheduler_config=scheduler_config_dropout
)

# Plot loss curves
plot_loss_curves(train_losses_dropout, val_accuracies_dropout, 'task2_dropout_loss_curve.png')

# Evaluate on test set
print("\nEvaluating Dropout_Net on test set...")
test_accuracy_dropout = predict(testloader, dropout_net)

# Comparison with baseline model
print("\n=== Task2 Results Comparison ===")
print("Dropout + L2 Regularization Model:")
if val_accuracies_dropout:
    print(f"  - Final validation accuracy: {val_accuracies_dropout[-1]:.2f}%")
else:
    print("  - Final validation accuracy: N/A")
print(f"  - Test accuracy: {test_accuracy_dropout:.2f}%")

# Task3: 调整参数
在`3. 模型训练与测试过程`部分中，我们定义了一些超参数（如`num_epoch`、优化器的`lr`）。调节这些参数，观察损失函数以及模型在测试集上的性能变化，在报告中简要说明这些指标的变化，尝试分析这些超参数对整个模型的影响。

In [ ]:
# Task3: Hyperparameter Tuning Experiments
import itertools
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import copy

# Hyperparameter grid (based on DEVELOP_PLAN.md)
hyperparams = {
    'lr': [0.01, 0.001],
    'weight_decay': [0, 1e-4],
    'label_smoothing': [0, 0.05]
}
batch_size = 64  # Fixed, consistent with earlier

print("="*70)
print("Task3: Hyperparameter Tuning Experiments")
print(f"Hyperparameter grid: {hyperparams}")
print(f"Batch size: {batch_size} (fixed)")
print(f"Total experiments: {len(hyperparams['lr']) * len(hyperparams['weight_decay']) * len(hyperparams['label_smoothing'])}")
print("="*70)

# Results storage
results = []

# Device setup
device = get_device()
print(f"Using device: {device}\n")

# Iterate through all hyperparameter combinations
for exp_idx, (lr, wd, ls) in enumerate(itertools.product(
    hyperparams['lr'], hyperparams['weight_decay'], hyperparams['label_smoothing']
), 1):
    print(f"\n{'='*60}")
    print(f"Experiment {exp_idx}/8: lr={lr}, weight_decay={wd}, label_smoothing={ls}")
    print(f"{'='*60}")
    
    # Create new data loader (using same transform)
    trainloader_exp = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
    
    # Initialize new model (LeNet)
    model = Net()
    model = model.to(device)
    
    # Configure optimizer (Adam, per development plan)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    
    # Configure loss function (with label smoothing)
    criterion = nn.CrossEntropyLoss(label_smoothing=ls)
    
    # Create learning rate scheduler (linear warm-up + cosine annealing with restarts)
    scheduler_config = create_learning_rate_scheduler(
        optimizer, scheduler_type='cosine',
        total_epochs=num_epochs, initial_lr=lr
    )
    
    # Train model (using my_utils.train_model)
    save_path = f'checkpoints/task3/exp_{exp_idx}'
    train_losses, val_accuracies = train_model(
        model=model,
        train_loader=trainloader_exp,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=num_epochs,
        save_path=save_path,
        val_loader=valloader,
        early_stopping_patience=10,
        gradient_clip=1.0,
        print_every=1000,
        scheduler_config=scheduler_config
    )
    
    # Record results
    final_val_acc = val_accuracies[-1] if val_accuracies else 0
    final_train_loss = train_losses[-1] if train_losses else float('inf')
    best_val_acc = max(val_accuracies) if val_accuracies else 0
    epochs_trained = len(train_losses)
    
    results.append({
        'experiment_id': exp_idx,
        'learning_rate': lr,
        'weight_decay': wd,
        'label_smoothing': ls,
        'batch_size': batch_size,
        'final_val_accuracy': final_val_acc,
        'final_train_loss': final_train_loss,
        'best_val_accuracy': best_val_acc,
        'epochs_trained': epochs_trained,
        'optimizer': 'Adam'
    })
    
    # Save checkpoint
    torch.save(model.state_dict(), f'checkpoints/task3_lr{lr}_wd{wd}_ls{ls}.pth')
    print(f"  Best validation accuracy: {best_val_acc:.2f}%, Final validation accuracy: {final_val_acc:.2f}%")
    print(f"  Epochs trained: {epochs_trained}/{num_epochs}")

# Create results DataFrame
results_df = pd.DataFrame(results)

print("\n" + "="*70)
print("Hyperparameter Tuning Results (sorted by best validation accuracy):")
print("="*70)
print(results_df.sort_values('best_val_accuracy', ascending=False).to_string())

# Save results to CSV
results_df.to_csv('task3_hyperparameter_results.csv', index=False)
print(f"\nResults saved to 'task3_hyperparameter_results.csv'")

# Visualize results
import matplotlib.pyplot as plt

# Plot effects of different hyperparameters
plt.figure(figsize=(15, 5))

# Subplot 1: Effect of learning rate
plt.subplot(1, 3, 1)
for lr in hyperparams['lr']:
    lr_data = results_df[results_df['learning_rate'] == lr]
    plt.bar([str(x) for x in lr_data['experiment_id']], lr_data['best_val_accuracy'], 
            alpha=0.7, label=f'lr={lr}')
plt.xlabel('Experiment ID')
plt.ylabel('Best Validation Accuracy (%)')
plt.title('Effect of Learning Rate')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 2: Effect of weight decay
plt.subplot(1, 3, 2)
for wd in hyperparams['weight_decay']:
    wd_data = results_df[results_df['weight_decay'] == wd]
    if len(wd_data) > 0:
        plt.bar([str(x) for x in wd_data['experiment_id']], wd_data['best_val_accuracy'],
                alpha=0.7, label=f'wd={wd}')
plt.xlabel('Experiment ID')
plt.ylabel('Best Validation Accuracy (%)')
plt.title('Effect of Weight Decay')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 3: Effect of label smoothing
plt.subplot(1, 3, 3)
for ls in hyperparams['label_smoothing']:
    ls_data = results_df[results_df['label_smoothing'] == ls]
    if len(ls_data) > 0:
        plt.bar([str(x) for x in ls_data['experiment_id']], ls_data['best_val_accuracy'],
                alpha=0.7, label=f'ls={ls}')
plt.xlabel('Experiment ID')
plt.ylabel('Best Validation Accuracy (%)')
plt.title('Effect of Label Smoothing')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task3_hyperparameter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Output best hyperparameter combination
best_exp = results_df.loc[results_df['best_val_accuracy'].idxmax()]
print("\n" + "="*70)
print("Best Hyperparameter Combination:")
print("="*70)
print(f"  Experiment ID: {best_exp['experiment_id']}")
print(f"  Learning Rate: {best_exp['learning_rate']}")
print(f"  Weight Decay: {best_exp['weight_decay']}")
print(f"  Label Smoothing: {best_exp['label_smoothing']}")
print(f"  Best Validation Accuracy: {best_exp['best_val_accuracy']:.2f}%")
print(f"  Final Validation Accuracy: {best_exp['final_val_accuracy']:.2f}%")
print(f"  Epochs Trained: {best_exp['epochs_trained']}")
print("="*70)

# Task4: 实现自己的网络
查阅资料（参考：[动手学深度学习](https://zh.d2l.ai/chapter_convolutional-modern/index.html)以及[`torchvision`的模型源码](https://github.com/pytorch/vision/tree/main/torchvision/models)），修改`2. 定义用于分类的网络结构`中的代码，实现一种现代卷积神经网络。与最基础的LeNet相比，你实现的神经网络在性能、训练时间上有何差异？

In [ ]:
# Task4: Implement Modern CNN (manual ResNet-18)
import torch
import torch.nn as nn
import torch.nn.functional as F

class BasicBlock(nn.Module):
    """Basic residual block for ResNet-18."""
    expansion = 1
    
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, downsample: Optional[nn.Module] = None) -> None:
        super().__init__()
        # First convolution
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        # Second convolution
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample
        self.stride = stride
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        if self.downsample is not None:
            identity = self.downsample(x)
            
        out += identity
        out = F.relu(out)
        
        return out

class MyCNN(nn.Module):
    """Manually implemented ResNet-18 for CIFAR-10."""
    
    def __init__(self, num_classes: int = 10, dropout_rate: float = 0.3) -> None:
        super().__init__()
        
        # Initial convolution adapted for 32x32 CIFAR-10 images
        # Original ResNet-18 uses 7x7 stride=2, but we use 3x3 stride=1 for small images
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        # ResNet-18 layers: 4 stages with [2, 2, 2, 2] basic blocks
        self.in_channels = 64
        
        # Stage 1: 64 -> 64, no downsampling in first block
        self.layer1 = self._make_layer(64, 2, stride=1)
        # Stage 2: 64 -> 128, downsampling in first block
        self.layer2 = self._make_layer(128, 2, stride=2)
        # Stage 3: 128 -> 256, downsampling in first block
        self.layer3 = self._make_layer(256, 2, stride=2)
        # Stage 4: 256 -> 512, downsampling in first block
        self.layer4 = self._make_layer(512, 2, stride=2)
        
        # Global average pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Dropout before classifier
        self.dropout = nn.Dropout(p=dropout_rate)
        
        # Final classifier
        self.fc = nn.Linear(512 * BasicBlock.expansion, num_classes)
        
    def _make_layer(self, out_channels: int, blocks: int, stride: int = 1) -> nn.Sequential:
        """Create a layer with specified number of basic blocks."""
        downsample = None
        
        # Create downsample layer if stride != 1 or channels change
        if stride != 1 or self.in_channels != out_channels * BasicBlock.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * BasicBlock.expansion,
                         kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * BasicBlock.expansion)
            )
            
        layers = []
        # First block in the layer (may have downsampling)
        layers.append(BasicBlock(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * BasicBlock.expansion
        
        # Remaining blocks in the layer (no downsampling)
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_channels, out_channels))
            
        return nn.Sequential(*layers)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Initial convolution
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        # Note: No maxpool here to preserve spatial dimensions for 32x32 input
        
        # Residual stages
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        # Global average pooling
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        
        # Dropout before classifier
        x = self.dropout(x)
        x = self.fc(x)
        
        return x

# Create MyCNN instance and view parameter count
device = get_device()
mycnn = MyCNN(num_classes=10, dropout_rate=0.3)
mycnn = mycnn.to(device)

param_counts = count_parameters(mycnn)

print("="*70)
print("Task4: Modern Convolutional Neural Network (Manual ResNet-18)")
print("="*70)
print(mycnn)
print(f"\nTotal parameters: {param_counts['total']:,}")
print(f"Trainable parameters: {param_counts['trainable']:,}")
print(f"Using device: {device}")

# Train MyCNN
print("\nStarting MyCNN training...")

# Use best hyperparameters from Task3 (or defaults)
optimizer_mycnn = torch.optim.Adam(mycnn.parameters(), lr=0.001, weight_decay=0.001)
criterion_mycnn = nn.CrossEntropyLoss(label_smoothing=0.05)

# Increase epochs as modern CNNs need more iterations to converge
num_epochs_mycnn = 50

# Create learning rate scheduler (linear warm-up + cosine annealing with restarts, T_0=25 for half of 50 epochs)
scheduler_config_mycnn = create_learning_rate_scheduler(
    optimizer_mycnn, scheduler_type='cosine',
    total_epochs=num_epochs_mycnn, initial_lr=0.001,
    T_0=25, T_mult=2
)

# Train model (using my_utils.train_model)
save_path_mycnn = 'checkpoints/mycnn_resnet18'
train_losses_mycnn, val_accuracies_mycnn = train_model(
    model=mycnn,
    train_loader=trainloader,
    criterion=criterion_mycnn,
    optimizer=optimizer_mycnn,
    num_epochs=num_epochs_mycnn,
    save_path=save_path_mycnn,
    val_loader=valloader,
    early_stopping_patience=15,
    gradient_clip=1.0,
    print_every=1000,
    scheduler_config=scheduler_config_mycnn
)

# Plot loss curves
plot_loss_curves(train_losses_mycnn, val_accuracies_mycnn, 'task4_mycnn_loss_curve.png')

# Evaluate on test set
print("\nEvaluating MyCNN on test set...")
test_accuracy_mycnn = predict(testloader, mycnn)

# Comparison with baseline model
print("\n" + "="*70)
print("Task4 Results: MyCNN (ResNet-18) vs Baseline LeNet")
print("="*70)
print("MyCNN (ResNet-18):")
if val_accuracies_mycnn:
    print(f"  - Best validation accuracy: {max(val_accuracies_mycnn):.2f}%")
    print(f"  - Final validation accuracy: {val_accuracies_mycnn[-1]:.2f}%")
else:
    print("  - Best validation accuracy: N/A")
    print("  - Final validation accuracy: N/A")
print(f"  - Test accuracy: {test_accuracy_mycnn:.2f}%")
print(f"  - Epochs trained: {len(train_losses_mycnn)}")
print(f"  - Total parameters: {param_counts['total']:,}")

print("\nAnalysis:")
print("1. Parameter count: MyCNN (~11M) is far larger than LeNet (~60K)")
print("2. Training time: MyCNN requires more time, but accuracy should be significantly higher")
print("3. Regularization: MyCNN includes BatchNorm and Dropout to help prevent overfitting")
print("4. Expected: MyCNN should achieve 85-90% accuracy on CIFAR-10")
print("="*70)